In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import average_precision_score, precision_recall_curve

In [ ]:
in_dir = "../../results/32_chrombpnet/14_variant_score_prediction"
out_dir = "../../results/32_chrombpnet/16_eval_variant_score"

os.makedirs(out_dir, exist_ok=True)

In [ ]:
def compute_average_precision(df):
    """
    Compute the average precision score for a given DataFrame.
    The DataFrame should contain 'y_true' and 'y_score' columns.
    """
    ap = average_precision_score(y_score=df["abs_logfc"],
                                 y_true=df["label"])
    
    precision, recall, _ = precision_recall_curve(y_true=df["label"],
                                                  y_score=df["abs_logfc"],
                                                  drop_intermediate=True)
    
    df_pr = pd.DataFrame({"precision": precision, "recall": recall})
    
    return ap, df_pr

In [ ]:
# read caQTLs
df_caqtl = pd.read_csv("../../data/Liver_caQTL/caQTL_variants_overlappingPeaks_LD-r2-0.8_withLead.bed", 
                       sep="\t")
df_caqtl["variant_id"] = df_caqtl["proxy_ID"] + ":1"
df_caqtl = df_caqtl[["variant_id", "proxy_caQTL_beta"]]
df_caqtl.rename(columns={"proxy_caQTL_beta": "beta"}, inplace=True)

In [ ]:
sns.histplot(df_caqtl['beta'], bins=50)

In [ ]:
df_caqtl['label'] = df_caqtl['beta'].apply(lambda x: 1 if abs(x) > 1 else 0)

In [ ]:
df_caqtl['label'].value_counts()

In [ ]:
df_caqtl

In [ ]:
# read the data
df_atac = pd.read_csv(f"{in_dir}/ATAC.variant_scores.tsv", sep="\t")
df_atac = df_atac[df_atac["variant_id"].isin(df_caqtl["variant_id"])].reset_index(drop=True)
df_atac = pd.merge(df_atac, df_caqtl, on="variant_id")

In [ ]:
ap, df_pr = compute_average_precision(df_atac)

In [ ]:
# plot precision-recall curve and add average precision score
plt.figure(figsize=(8, 6))
plt.title(f"Precision-Recall Curve (AP={ap:.3f})")
plt.xlabel("Recall")
plt.ylim(0, 1)
plt.ylabel("Precision")
plt.grid(True)
plt.plot(df_pr["recall"], df_pr["precision"], label="ATAC")
plt.legend()

In [ ]:
# read the data
df_access = pd.read_csv(f"{in_dir}/ACCESS.variant_scores.tsv", sep="\t")
df_access = df_access[df_access["variant_id"].isin(df_caqtl["variant_id"])].reset_index(drop=True)
df_access = pd.merge(df_access, df_caqtl, on="variant_id")

In [ ]:
ap, df_pr = compute_average_precision(df_access)

In [ ]:
# plot precision-recall curve and add average precision score
plt.figure(figsize=(8, 6))
plt.title(f"Precision-Recall Curve (AP={ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.plot(df_pr["recall"], df_pr["precision"], label="ACCESS")
plt.legend()

In [ ]:
# add R squared
pearsonR, pv = stats.pearsonr(df_atac['beta'], df_atac['logfc'])

ax = sns.jointplot(
    data=df_atac,
    x="beta",
    y="logfc",
    kind = 'scatter',
    joint_kws={'marker':'o', 's':10, 'alpha':0.1, 'linewidth':0},
    marginal_kws={'bins':20, 'element':'step', 'kde':True, 'linewidth':0},
)
ax.plot_joint(sns.regplot, color="r", scatter=False, 
              line_kws={"color": "orange", 'linestyle':'dashed'})

plt.xlim(-1.5, 1.5)
plt.ylim(-0.5, 0.5)
plt.title('pearsonR = {:.3f}'.format(pearsonR))
plt.xlabel('Observed effect size (caQTL beta)')
plt.ylabel('Predicted effect size (logfc)')
plt.tight_layout()

In [ ]:
# add R squared
pearsonR, pv = stats.pearsonr(df_access['beta'], 
                              df_access['logfc'])

ax = sns.jointplot(
    data=df_access,
    x="beta",
    y="logfc",
    kind = 'scatter',
    joint_kws={'marker':'o', 's':10, 'alpha':0.1, 'linewidth':0},
    marginal_kws={'bins':20, 'element':'step', 'kde':True, 'linewidth':0},
)
ax.plot_joint(sns.regplot, color="r", scatter=False, 
              line_kws={"color": "orange", 'linestyle':'dashed'})

plt.xlim(-1.5, 1.5)
plt.ylim(-0.5, 0.5)
plt.title('pearsonR = {:.3f}'.format(pearsonR))
plt.xlabel('Observed effect size (caQTL beta)')
plt.ylabel('Predicted effect size (logfc)')
plt.tight_layout()

In [ ]:
df_atac.to_csv(f"{out_dir}/ATAC.variant_scores_with_caQTL_labels.tsv", sep="\t", index=False)
df_access.to_csv(f"{out_dir}/ACCESS.variant_scores_with_caQTL_labels.tsv", sep="\t", index=False)